# Surgical Patient Flow Intelligence — Part 2
**Continued from Part 1:** Setup, EDA, Feature Engineering, and Model 1 (Discharge Readiness)

This notebook covers:
- **Model 2** — Length of Stay (LOS) Prediction
- **Model 3** — 30-Day Readmission Risk
- **Task 4** — Clinical Intervention Recommendations


## <a id="model-2-los-prediction"></a> MODEL 2 - LOS Prediction

### <a id="admission-los-model"></a> 1. Admission LOS Model

In [ ]:
los_adm_df = master_df_2.drop_duplicates(subset=['encounter_id']).copy()

leakage_cols = [
    'discharge_ready_day',
    'length_from_admission',
    'recovery_score'
]
los_adm_df = los_adm_df.drop(columns=leakage_cols, errors='ignore')

target = 'actual_los_days'
X_adm = los_adm_df.drop(columns=[target, 'encounter_id'], errors='ignore')
y_adm = los_adm_df[target]

print(X_adm.shape, y_adm.shape)

#### <a id="train-model-admission"></a> Train Model

In [ ]:
from lightgbm import LGBMRegressor

X_train, X_test, y_train, y_test = train_test_split(
    X_adm, y_adm, test_size=0.3, random_state=42
)

model_adm = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)
model_adm.fit(X_train, y_train)

#### <a id="evaluation-admission"></a> Evaluation

In [ ]:
y_pred = model_adm.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("ADMISSION LOS MODEL")
print(f"MAE  : {mae:.3f} days")
print(f"RMSE : {rmse:.3f} days")
print(f"\nMean Actual LOS    : {y_test.mean():.2f}")
print(f"Mean Predicted LOS : {y_pred.mean():.2f}")

### <a id="part-2-post-operative-model"></a> PART 2 POST-OPERATIVE MODEL

In [ ]:
los_dyn_df = master_df_2.copy()
los_dyn_df = los_dyn_df.drop(
    columns=[
        'discharge_ready_day',
        'length_from_admission',
        'recovery_score'
    ],
    errors='ignore'
)

target = 'actual_los_days'
X_dyn = los_dyn_df.drop(columns=[target, 'encounter_id'], errors='ignore')
y_dyn = los_dyn_df[target]

print(X_dyn.shape, y_dyn.shape)

In [ ]:
X_train_dyn, X_test_dyn, y_train_dyn, y_test_dyn = train_test_split(
    X_dyn, y_dyn, test_size=0.3, random_state=42
)

model_dyn = LGBMRegressor(
    n_estimators=400,
    learning_rate=0.03,
    max_depth=6,
    random_state=42
)
model_dyn.fit(X_train_dyn, y_train_dyn)

#### <a id="evaluation-post-operative"></a> Evaluate

In [ ]:
y_pred_dyn = model_dyn.predict(X_test_dyn)
mae_dyn = mean_absolute_error(y_test_dyn, y_pred_dyn)
rmse_dyn = np.sqrt(mean_squared_error(y_test_dyn, y_pred_dyn))

print("POST-OPERATIVE LOS MODEL")
print(f"MAE  : {mae_dyn:.3f} days")
print(f"RMSE : {rmse_dyn:.3f} days")
print(f"\nMean Actual LOS    : {y_test_dyn.mean():.2f}")
print(f"Mean Predicted LOS : {y_pred_dyn.mean():.2f}")

In [ ]:
feature_imp = pd.DataFrame({
    'feature': X_dyn.columns,
    'importance': model_dyn.feature_importances_
}).sort_values(by='importance', ascending=False)

plt.figure(figsize=(10,6))
plt.barh(feature_imp['feature'][:15], feature_imp['importance'][:15])
plt.gca().invert_yaxis()
plt.title("Top Features for LOS Prediction (Dynamic Model)")
plt.show()

In [ ]:
residuals = y_test_dyn - y_pred_dyn
plt.figure(figsize=(6,4))
plt.hist(residuals, bins=30)
plt.title("Residual Distribution")
plt.xlabel("Error (days)")
plt.ylabel("Frequency")
plt.show()

### <a id="actual-vs-predicted-los"></a> ACTUAL VS PREDICTED

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_test_dyn, y_pred_dyn, alpha=0.5)
plt.plot(
    [y_test_dyn.min(), y_test_dyn.max()],
    [y_test_dyn.min(), y_test_dyn.max()],
    color='red'
)
plt.xlabel("Actual LOS")
plt.ylabel("Predicted LOS")
plt.title("Actual vs Predicted LOS (Dynamic)")
plt.show()

### <a id="los-final-summary"></a> Final Summary Evaluation — Length of Stay (LOS) Prediction

## <a id="model-3-readmission-risk"></a> MODEL 3 - 30-Day Readmission Risk

In [ ]:
# Encounter-level dataset
readmit_df = master_df_2.drop_duplicates(subset=['encounter_id']).copy()
print(readmit_df.shape)
readmit_df['readmission_30d'].value_counts()

In [ ]:
drop_cols = [
    # 'postop_day',
    # 'vitals_stability_score',
    'discharge_ready_day',
    'length_from_admission',
    'recovery_score'
]
readmit_df = readmit_df.drop(columns=drop_cols, errors='ignore')
readmit_df['readmission_30d'].value_counts(normalize=True)

In [ ]:
### <a id="top-30-features-readmission"></a> top 30 features

In [ ]:
target = 'readmission_30d'
X = readmit_df.drop(columns=[target, 'encounter_id'], errors='ignore')
y = readmit_df[target]

print(X.shape, y.shape)
print("Readmission Rate:", y.mean())

#### <a id="train-test-split-readmission"></a> Train Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [ ]:
# pip install XGBOOST
readmit_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    class_weight='balanced',
    random_state=42
)



readmit_model.fit(X_train, y_train)

y_prob_readmit = readmit_model.predict_proba(X_test)[:, 1]

threshold = 0.30
y_pred_readmit = (y_prob_readmit >= threshold).astype(int)

#### <a id="evaluation-readmission"></a> Evaluation

In [ ]:
print("TASK C — READMISSION RISK")
print(f"AUROC : {roc_auc_score(y_test, y_prob_readmit):.4f}")
print(f"AUPRC : {average_precision_score(y_test, y_prob_readmit):.4f}")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_readmit, target_names=['No Readmit', 'Readmit']))

In [ ]:
#### <a id="decision-analysis-readmission"></a> Decision Analysis

In [ ]:
# Confusion Matrix for Model 3
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_readmit)
).plot(
    ax=ax,
    colorbar=False,
    cmap="Reds"
)
ax.set_title("30-Day Readmission Risk")
plt.tight_layout()
plt.show()

In [ ]:
  def readmission_risk(prob):
      if prob >= 0.45:
          return "High Risk"
      elif prob >= 0.30:
          return "Medium Risk"
      else:
          return "Low Risk"

  test_df = X_test.copy()
  test_df['probability'] = y_prob_readmit
  test_df['risk_group'] = test_df['probability'].apply(readmission_risk)
  print(test_df['risk_group'].value_counts())

In [ ]:
pd.crosstab(test_df['risk_group'], y_test)

In [ ]:
import shap
explainer = shap.TreeExplainer(readmit_model)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test)

In [ ]:
feature_imp = pd.DataFrame({
    'feature': X.columns,
    'importance': readmit_model.feature_importances_
}).sort_values(by='importance', ascending=False)

plt.figure(figsize=(10,6))
plt.barh(feature_imp['feature'][:15], feature_imp['importance'][:15])
plt.gca().invert_yaxis()
plt.title("Top Features for Readmission Risk")
plt.show()

In [ ]:
feature_imp

In [ ]:
from sklearn.metrics import (
    roc_curve,
    precision_recall_curve,
    roc_auc_score,
    average_precision_score
)

# ROC + Precision-Recall Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    "Readmission Risk — ROC & Precision-Recall",
    fontsize=13,
    fontweight="bold"
)

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob_readmit)
axes[0].plot(fpr, tpr, lw=2, color='red', label=f"AUC = {roc_auc_score(y_test, y_prob_readmit):.3f}")
axes[0].plot([0, 1], [0, 1], "k--", lw=1)
axes[0].set_title("ROC Curve")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend()

# Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_test, y_prob_readmit)
axes[1].plot(recall, precision, lw=2, color='red', label=f"AUPRC = {average_precision_score(y_test, y_prob_readmit):.3f}")
axes[1].axhline(y_test.mean(), color="k", linestyle="--", label=f"Baseline = {y_test.mean():.2f}")
axes[1].set_title("Precision-Recall Curve")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Feature Importance for Readmission Model
readmit_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': readmit_model.feature_importances_
}).sort_values(by='importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(
    data=readmit_importance.head(15),
    x='importance',
    y='feature',
    palette='Reds_r'
)
plt.title('Top 15 Drivers of 30-Day Readmission Risk', fontsize=13, fontweight='bold')
plt.xlabel('XGBoost Feature Importance')
plt.ylabel('Clinical/Operational Feature')
plt.tight_layout()
plt.show()

### <a id="readmission-model-final-summary"></a> Readmission Model — Final Summary

## <a id="task-4-clinical-intervention-recommendations"></a> Task 4: Clinical Intervention Recommendations

### <a id="summary-of-task-4-recommendations"></a> Summary of Task 4 Recommendations

In [ ]:
all_rec_list = []
for recs in recommendation_view['recommended_interventions']:
    all_rec_list.extend(recs.split(" | "))

rec_counts = pd.Series(all_rec_list).value_counts().sort_values(ascending=True)

# Plotting using horizontal bar chart
plt.figure(figsize=(10, 6))
rec_counts.plot(kind='barh', color='teal')

plt.title("Frequency of Clinical Recommendations (Test Set)", fontsize=14, fontweight='bold')
plt.xlabel("Count of Recommendations")
plt.ylabel("Intervention Type")
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
def generate_recommendations(row):
    recommendations = []

    # 1. High Readmission Risk Intervention
    if row.get('readmission_prob', 0) > 0.4:
        recommendations.append("High-Touch Post-Discharge Follow-up (72 hrs)")
        if row.get('social_barrier_flag') == 1:
            recommendations.append("Social Work / Care Coordination Referral")

    # 2. LOS Reduction (Delayed Discharge Readiness)
    if row.get('discharge_ready_prob', 1) < 0.5:
        if row.get('mobility_score', 100) < 50:
            recommendations.append("Escalate Physical Therapy / Mobility Goals")
        if row.get('pain_score', 0) > 6:
            recommendations.append("Multimodal Pain Management Review")
        if row.get('care_plan_complete') == 0:
            recommendations.append("Prioritize Clinical Care Plan Completion")

    # 3. Operational Quality of Care
    icu = row.get('icu_required', 0)
    vitals = row.get('vitals_stability_score', 100)
    if icu == 1 and vitals < 70:
        recommendations.append("Critical Care Specialist Consultation")

    if not recommendations:
        recommendations.append("Continue Standard ERAS Protocol")

    return " | ".join(recommendations)


current_readmission_test_indices = X_test.index

discharge_model_excluded_cols = [
    'recovery_score', 'discharge_ready_day', 'actual_los_days',
    'length_from_admission', 'vitals_stability_score',
    'care_plan_complete', 'postop_day', 'discharge_ready_label'
]


full_discharge_X = master_df_2.drop(columns=discharge_model_excluded_cols, errors='ignore')


features_for_prediction_raw = full_discharge_X.loc[current_readmission_test_indices]


train_features = model.feature_name_


features_for_prediction = features_for_prediction_raw.reindex(columns=train_features, fill_value=0)


recommendation_view = master_df_2.loc[current_readmission_test_indices].copy()

recommendation_view['readmission_prob'] = y_prob_readmit


recommendation_view['discharge_ready_prob'] = model.predict_proba(features_for_prediction)[:, 1]


recommendation_view['recommended_interventions'] = recommendation_view.apply(generate_recommendations, axis=1)


display(recommendation_view[['readmission_prob', 'discharge_ready_prob', 'recommended_interventions']].head(10))

In [ ]:
all_rec_list = []
for recs in recommendation_view['recommended_interventions']:
    all_rec_list.extend(recs.split(" | "))

rec_counts = pd.Series(all_rec_list).value_counts()

plt.figure(figsize=(10, 6))
sns.barplot(x=rec_counts.values, y=rec_counts.index, palette='viridis')
plt.title("Frequency of Clinical Recommendations (Test Set)")
plt.xlabel("Count")
plt.ylabel("Intervention Type")
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Select and export the relevant clinical columns for validation
export_cols = [
    'readmission_prob',
    'discharge_ready_prob',
    'recommended_interventions'
]

output_filename = 'surgical_clinical_recommendations.csv'
recommendation_view[export_cols].to_csv(output_filename, index=True)
print(f'Successfully exported recommendations to {output_filename}')

In [ ]:
recommendation_view.head()

In [ ]:
excel_file = '/content/surgical_clinical_recommendations.xlsx'
df_excel = pd.read_excel(excel_file)
display(df_excel.head())